# Retriever upgrade on COVID-QA: BM25 → dense → RRF → reranker (starter)

The chunking notebooks showed the chunker is not the bottleneck: 94% of answers sit whole
in some chunk, but BM25 finds the right one in the top 5 for only about 68% of questions.
This notebook builds the production retrieval stack one stage at a time on the same
chunks and questions, so each row shows what one stage adds.

```
question ─┬─► BM25 (words)   top 50 ─┐
          │                          ├─► RRF merge ─► reranker reads (question, chunk) ─► top 5
          └─► dense (meaning) top 50 ┘
```

| Row | Adds | Runs here with |
|---|---|---|
| 1 | BM25 | rank_bm25 |
| 2 | dense | `BAAI/bge-small-en-v1.5` with its query prefix (CPU) |
| 3 | RRF fusion of 1 + 2 | plain Python |
| 4 | cross-encoder reranker on the RRF top 20 | `cross-encoder/ms-marco-MiniLM-L-6-v2` on a 300-question sample (CPU) |

## Capabilities

| Name | What it does | Example |
|---|---|---|
| `bm25_rank` / `dense_rank` | top-50 chunk indices per question | `[[812, 77, ...], ...]` |
| `rrf` | merge two ranked lists by rank position | `rrf(a, b)[:5]` |
| `rerank` | re-score the top-n with a cross-encoder | `[77, 812, ...]` |
| `hits_at` | 1/0 per question: right paper + gold answer in the top k | `[1, 0, 1, ...]` |
| `report` | paired bootstrap CI between two rows | `+3.1 pts [+1.9, +4.3] real gain` |

Chunks: labelled Docling 512 (`Paper: <title>` + heading path + text). If
`04-benchmarks/covid-qa/data/chunks/docling_512_label.jsonl` exists (written by
`maanas-chunking-eval/04-...ipynb`) it is loaded; otherwise the chunks are built here.
Offline once the data and models are on disk; no API key.

### Step 1 · Bootstrap the environment

In [1]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()
REPO = _root
print("repository root found")

repository root found


### Step 2 · Load the gold questions (downloads from Hugging Face if missing)

In [2]:
import json
import tarfile

DATA = REPO / "04-benchmarks" / "covid-qa" / "data"
PARSED = DATA / "02_parsed" / "covidqa"
GOLD = DATA / "gold" / "questions.jsonl"
CHUNK_DIR = DATA / "chunks"          # gitignored with the rest of data/


def ensure_data():
    if GOLD.is_file() and PARSED.is_dir():
        return
    import zstandard as zstd
    from huggingface_hub import hf_hub_download
    DATA.mkdir(parents=True, exist_ok=True)
    hf_hub_download("anacodicai-labs/covidqa-bench", "gold/questions.jsonl",
                    repo_type="dataset", local_dir=str(DATA))
    archive = hf_hub_download("anacodicai-labs/covidqa-bench", "parsed_questions_137.tar.zst",
                              repo_type="dataset", local_dir=str(DATA))
    with open(archive, "rb") as fh:
        reader = zstd.ZstdDecompressor().stream_reader(fh)
        with tarfile.open(fileobj=reader, mode="r|") as tar:
            tar.extractall(DATA)


ensure_data()
questions = []
for line in open(GOLD, encoding="utf-8"):
    if line.strip():
        questions.append(json.loads(line))
paper_ids = sorted({q["paper_id"] for q in questions})
print(len(questions), "questions over", len(paper_ids), "papers")

1852 questions over 137 papers


### Step 3 · Text normaliser (same as the chunking notebooks)

In [3]:
import re
import unicodedata


def norm(s):
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = re.sub(r"-\s*\n\s*", "", s)
    return re.sub(r"[^a-z0-9]+", " ", s).strip()


print(norm("Fourteen days."))

fourteen days


### Step 4 · Load the shared chunks, or build them

In [4]:
from docling.chunking import HybridChunker
from docling_core.types.doc import DoclingDocument


def paper_title(doc):
    for item in doc.texts:
        label = getattr(item.label, "value", str(item.label))
        if label == "title" and item.text.strip():
            return item.text.strip()
    return doc.name


def load_doc(pid):
    return DoclingDocument.load_from_json(PARSED / pid / (pid + ".docling.json"))


def docling_chunks(doc, pid, max_tokens, labelled):
    chunker = HybridChunker(max_tokens=max_tokens)
    title = paper_title(doc)
    rows = []
    for ch in chunker.chunk(doc):
        headings = []
        if ch.meta.headings:
            for h in ch.meta.headings:
                headings.append(h)
        index_text = ch.text
        if labelled:
            index_text = "Paper: " + title + "\n" + chunker.contextualize(ch)
        rows.append({"paper": pid, "section": " > ".join(headings),
                     "text": ch.text, "index_text": index_text})
    return rows

In [5]:
import time

shared = CHUNK_DIR / "docling_512_label.jsonl"
chunks = []
if shared.is_file():
    for line in open(shared, encoding="utf-8"):
        chunks.append(json.loads(line))
    print("loaded", len(chunks), "chunks from", shared.relative_to(REPO))
else:
    t0 = time.time()
    for pid in paper_ids:
        chunks.extend(docling_chunks(load_doc(pid), pid, 512, labelled=True))
    print("built", len(chunks), "chunks in", round(time.time() - t0), "s")

loaded 5912 chunks from 04-benchmarks/covid-qa/data/chunks/docling_512_label.jsonl


### Step 5 · Scoring functions

In [6]:
def hits_at(chunks, ranked, k=5, qs=None):
    """1 if a top-k chunk is from the question's paper and contains the gold answer."""
    if qs is None:
        qs = list(range(len(questions)))
    texts = []
    for c in chunks:
        texts.append(norm(c["text"]))
    out = []
    for qi in qs:
        q = questions[qi]
        ans = norm(q["gold_answer"])
        hit = 0
        for j in ranked[qi][:k]:
            if chunks[j]["paper"] == q["paper_id"] and ans and ans in texts[j]:
                hit = 1
                break
        out.append(hit)
    return out

In [7]:
import random


def paired_ci(a, b, reps=2000, seed=0):
    rng = random.Random(seed)
    n = len(a)
    diffs = []
    for _ in range(reps):
        s = 0
        for _ in range(n):
            k = rng.randrange(n)
            s += b[k] - a[k]
        diffs.append(s / n)
    diffs.sort()
    observed = sum(b[k] - a[k] for k in range(n)) / n
    return observed, diffs[int(0.025 * reps)], diffs[int(0.975 * reps)]


def report(name_a, a, name_b, b):
    d, lo, hi = paired_ci(a, b)
    verdict = "real gain" if lo > 0 else ("real loss" if hi < 0 else "tie (CI crosses 0)")
    print(f"{name_b} vs {name_a}: {d * 100:+.1f} pts  95% CI [{lo * 100:+.1f}, {hi * 100:+.1f}]  {verdict}")

## Row 1 · BM25

In [8]:
from rank_bm25 import BM25Okapi


def bm25_rank(chunks, k=50):
    """Top-k chunk indices per question with BM25 over index_text."""
    corpus = []
    for c in chunks:
        corpus.append(norm(c["index_text"]).split())
    bm25 = BM25Okapi(corpus)
    ranked = []
    for q in questions:
        scores = bm25.get_scores(norm(q["question"]).split())
        top = sorted(range(len(scores)), key=lambda j: -scores[j])[:k]
        ranked.append(top)
    return ranked

In [9]:
r_bm25 = bm25_rank(chunks)
h_bm25 = hits_at(chunks, r_bm25)
print(f"BM25 Recall@5 {sum(h_bm25) / len(questions):.1%}")

BM25 Recall@5 68.2%


## Row 2 · Dense

bge models expect an instruction in front of the **query** only. Leaving it out is a
common silent loss, so the prefix is a parameter here.

In [10]:
from sentence_transformers import SentenceTransformer


def dense_rank(chunks, model_name, query_prefix="", doc_prefix="", k=50, batch_size=32):
    """Top-k chunk indices per question by cosine similarity."""
    model = SentenceTransformer(model_name)
    doc_texts = []
    for c in chunks:
        doc_texts.append(doc_prefix + c["index_text"])
    q_texts = []
    for q in questions:
        q_texts.append(query_prefix + q["question"])
    D = model.encode(doc_texts, batch_size=batch_size, normalize_embeddings=True,
                     show_progress_bar=False, convert_to_tensor=True)
    Q = model.encode(q_texts, batch_size=batch_size, normalize_embeddings=True,
                     show_progress_bar=False, convert_to_tensor=True)
    sims = Q @ D.T
    top = sims.topk(k, dim=1).indices.tolist()
    return top

In [11]:
DENSE_MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
t0 = time.time()
r_dense = dense_rank(chunks, DENSE_MODEL, query_prefix=QUERY_PREFIX)
h_dense = hits_at(chunks, r_dense)
print(f"{time.time() - t0:.0f}s | dense Recall@5 {sum(h_dense) / len(questions):.1%}")
report("bm25", h_bm25, "dense", h_dense)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

1334s | dense Recall@5 51.8%


dense vs bm25: -16.4 pts  95% CI [-18.7, -14.0]  real loss


**YOUR TURN (row 2).**
1. Rerun once with `QUERY_PREFIX = ""` and report the difference: that is the prefix check.
2. On Colab (free T4), swap in `Qwen/Qwen3-Embedding-0.6B` with the query instruction
   from its model card.

## Row 3 · RRF (reciprocal rank fusion)

Merge the BM25 and dense lists by **rank**, not by score: a chunk near the top of both
lists rises. `k = 60` is the usual constant.

In [12]:
def rrf(list_a, list_b, k=60):
    score = {}
    for rank in range(len(list_a)):
        cid = list_a[rank]
        score[cid] = score.get(cid, 0) + 1 / (k + rank + 1)
    for rank in range(len(list_b)):
        cid = list_b[rank]
        score[cid] = score.get(cid, 0) + 1 / (k + rank + 1)
    merged = []
    for cid in score:
        merged.append((score[cid], cid))
    merged.sort(reverse=True)
    result = []
    for s, cid in merged:
        result.append(cid)
    return result

In [13]:
r_rrf = []
for qi in range(len(questions)):
    r_rrf.append(rrf(r_bm25[qi], r_dense[qi])[:50])
h_rrf = hits_at(chunks, r_rrf)
print(f"RRF Recall@5 {sum(h_rrf) / len(questions):.1%}")
report("bm25", h_bm25, "rrf", h_rrf)
report("dense", h_dense, "rrf", h_rrf)

RRF Recall@5 66.3%


rrf vs bm25: -1.9 pts  95% CI [-3.7, -0.2]  real loss


rrf vs dense: +14.4 pts  95% CI [+12.6, +16.3]  real gain


## Row 4 · Reranker on the RRF top 20

A cross-encoder reads the question and one chunk **together** and scores the pair. It is
slow, so it only sees the top 20 from RRF, and on CPU it runs on a fixed random sample of
300 questions. Rows 3 and 4 are compared on that same sample.

In [14]:
from sentence_transformers import CrossEncoder

RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
RERANK_TOP = 20
SAMPLE = 300

reranker = CrossEncoder(RERANK_MODEL)


def rerank(qi, candidates):
    pairs = []
    for j in candidates:
        pairs.append((questions[qi]["question"], chunks[j]["text"]))
    scores = reranker.predict(pairs, show_progress_bar=False)
    order = sorted(range(len(candidates)), key=lambda t: -scores[t])
    result = []
    for t in order:
        result.append(candidates[t])
    return result

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [15]:
sample_qs = sorted(random.Random(0).sample(range(len(questions)), SAMPLE))
t0 = time.time()
r_rerank = {}
for qi in sample_qs:
    r_rerank[qi] = rerank(qi, r_rrf[qi][:RERANK_TOP])
print(f"reranked {SAMPLE} questions in {time.time() - t0:.0f}s")

ranked_sample = []
for qi in range(len(questions)):
    ranked_sample.append(r_rerank.get(qi, r_rrf[qi]))
h_rrf_s = hits_at(chunks, r_rrf, qs=sample_qs)
h_rerank_s = hits_at(chunks, ranked_sample, qs=sample_qs)
print(f"on the {SAMPLE}-question sample: RRF {sum(h_rrf_s) / SAMPLE:.1%} | + reranker {sum(h_rerank_s) / SAMPLE:.1%}")
report("rrf (sample)", h_rrf_s, "rerank (sample)", h_rerank_s)

reranked 300 questions in 781s


on the 300-question sample: RRF 68.0% | + reranker 77.7%


rerank (sample) vs rrf (sample): +9.7 pts  95% CI [+5.3, +14.0]  real gain


**YOUR TURN (row 4).** On Colab GPU, run all 1,852 questions with
`BAAI/bge-reranker-v2-m3` or `Qwen/Qwen3-Reranker-0.6B`, rerank the top 50, and report
latency per question next to the gain.

## Summary table

In [16]:
print(f"{'row':<28}{'Recall@5':>10}")
print(f"{'1 BM25':<28}{sum(h_bm25) / len(questions):>10.1%}")
print(f"{'2 dense (' + DENSE_MODEL.split('/')[-1] + ')':<28}{sum(h_dense) / len(questions):>10.1%}")
print(f"{'3 RRF (1 + 2)':<28}{sum(h_rrf) / len(questions):>10.1%}")
print(f"{'4 + reranker (sample)':<28}{sum(h_rerank_s) / SAMPLE:>10.1%}   (RRF on same sample: {sum(h_rrf_s) / SAMPLE:.1%})")
print("ceiling (answer whole in some chunk): about 94%")

row                           Recall@5
1 BM25                           68.2%
2 dense (bge-small-en-v1.5)      51.8%
3 RRF (1 + 2)                    66.3%
4 + reranker (sample)            77.7%   (RRF on same sample: 68.0%)
ceiling (answer whole in some chunk): about 94%


## Next, after the four rows

- **Paraphrased questions**: rewrite each question so it does not reuse the passage's
  words (an LLM on the free Groq tier can do this), then rerun rows 1-4. COVID-QA's
  question writers copied passage wording, which flatters BM25; real users paraphrase.
- **Filtered set**: drop questions that name no paper ("What was the purpose of this
  study?") and report both.

## Known gaps

- Row 2 and row 4 use small CPU models here; the numbers show the wiring and a lower bound.
- Row 4 is measured on a 300-question sample on CPU.
- COVID-QA is single-hop. Knowledge-graph and agentic retrieval help most on multi-hop
  questions, which need a different question set.